<a href="https://colab.research.google.com/github/smanjullee/MAURICYCLE/blob/main/MAURICYCLE_DASHBOARD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## MAURICYCLE AI BIN DASHBOARD

### Bin Status Legend:
*   **GREEN** - EMPTY BIN
*   **ORANGE** - HALF BIN
*   **RED** - FULL BIN

In [1]:
# Install necessary libraries
!pip install gradio pandas folium geopy

In [3]:
import gradio as gr
import pandas as pd
import folium
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# --- Configuration and Data Loading ---

# Attempt to load the logo image. Adjust path if necessary.
LOGO_PATH = 'newlogo.jpeg'

# Load the Excel file
try:
    df_gps = pd.read_excel('gps.xlsx')
except FileNotFoundError:
    print("Error: gps.xlsx not found. Please upload it to your Colab environment.")
    df_gps = pd.DataFrame({'Place': [], 'Weight(Kg)': []}) # Create an empty DataFrame to avoid errors

# Initialize geolocator with a user_agent
geolocator = Nominatim(user_agent="mauricycle_app")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

# Cache for geocoded locations to avoid repeated API calls
location_cache = {}

def get_coordinates(place_name):
    if place_name in location_cache:
        return location_cache[place_name]
    try:
        location = geocode(place_name + ", Mauritius") # Add Mauritius to improve accuracy
        if location:
            coords = (location.latitude, location.longitude)
            location_cache[place_name] = coords
            return coords
    except Exception as e:
        print(f"Error geocoding {place_name}: {e}")
    location_cache[place_name] = None # Cache failed attempts too
    return None

# Pre-defined list of towns/villages in Mauritius for the 'Source' dropdown
mauritius_towns = [
    "Port Louis", "Beau Bassin-Rose Hill", "Vacoas-Phoenix", "Curepipe",
    "Quatre Bornes", "Saint Pierre", "Centre de Flacq", "Mahebourg",
    "Goodlands", "Triolet", "Bel Air Rivière Sèche", "Grand Gaube"
]

# Get unique 'Place' names for the 'Destination' dropdown
destination_places = sorted(df_gps['Place'].unique().tolist())

Error: gps.xlsx not found. Please upload it to your Colab environment.


In [8]:
print("Columns in df_gps:", df_gps.columns.tolist())
print(df_gps.info())

Columns in df_gps: ['Device ID', 'Place', 'Latitude', 'Longitude', 'Timestamp (MUT)', 'Weight (kg)', 'Battery (%)', 'Solar Input (W)', 'Accuracy (m)']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Device ID        14 non-null     object 
 1   Place            14 non-null     object 
 2   Latitude         14 non-null     object 
 3   Longitude        14 non-null     float64
 4   Timestamp (MUT)  14 non-null     object 
 5   Weight (kg)      14 non-null     int64  
 6   Battery (%)      14 non-null     int64  
 7   Solar Input (W)  14 non-null     float64
 8   Accuracy (m)     14 non-null     float64
dtypes: float64(3), int64(2), object(4)
memory usage: 1.1+ KB
None


In [4]:
# --- Function to generate the map ---
def generate_bin_map(source=None, destination=None):
    # Initialize a map centered on Mauritius
    m = folium.Map(location=[-20.3, 57.5], zoom_start=10)

    # Add bin markers
    for index, row in df_gps.iterrows():
        place = row['Place']
        weight = row['Weight (kg)']
        coords = get_coordinates(place)

        if coords:
            lat, lon = coords
            color = 'green'  # Empty bin
            if weight >= 12 and weight <= 13:
                color = 'orange' # Half bin
            elif weight > 23:
                color = 'red'    # Full bin

            folium.CircleMarker(
                location=[lat, lon],
                radius=8,
                color=color,
                fill=True,
                fill_color=color,
                fill_opacity=0.7,
                tooltip=f"Place: {place}<br>Weight: {weight} Kg"
            ).add_to(m)

    # Add path if source and destination are selected
    if source and destination and source != destination:
        source_coords = get_coordinates(source)
        destination_coords = get_coordinates(destination)

        if source_coords and destination_coords:
            folium.PolyLine(
                locations=[source_coords, destination_coords],
                color='red',
                weight=5,
                opacity=0.8,
                tooltip=f"Path from {source} to {destination}"
            ).add_to(m)

    # Save map to an HTML file and return its content
    map_html_path = "mauritius_bins_map.html"
    m.save(map_html_path)
    with open(map_html_path, 'r') as f:
        map_html = f.read()
    return map_html


In [ ]:
# --- Gradio Interface Layout ---
with gr.Blocks(title="MAURICYCLE AI BIN DASHBOARD") as demo:
    # Display the logo if available
    if LOGO_PATH:
        try:
            gr.Image(LOGO_PATH, width=100, height=100, show_label=False, container=False)
        except Exception as e:
            gr.Markdown(f"<p style='color:red;'>Could not load logo: {e}</p>")

    gr.Markdown("# MAURICYCLE AI BIN DASHBOARD")
    gr.Markdown("### Bin Status Legend: GREEN - EMPTY BIN | ORANGE - HALF BIN | RED - FULL BIN")

    with gr.Row():
        source_dropdown = gr.Dropdown(
            label="Source",
            choices=mauritius_towns,
            value=mauritius_towns[0] if mauritius_towns else None,
            interactive=True
        )
        destination_dropdown = gr.Dropdown(
            label="Destination",
            choices=destination_places,
            value=destination_places[0] if destination_places else None,
            interactive=True
        )

    map_output = gr.HTML(label="Bin Locations and Path")

    # Initial map generation on load
    demo.load(generate_bin_map, inputs=[source_dropdown, destination_dropdown], outputs=map_output)

    # Update map when dropdowns change
    source_dropdown.change(
        generate_bin_map,
        inputs=[source_dropdown, destination_dropdown],
        outputs=map_output
    )
    destination_dropdown.change(
        generate_bin_map,
        inputs=[source_dropdown, destination_dropdown],
        outputs=map_output
    )

# Launch the Gradio app
demo.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d2639a3ea230aaa96f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
